# HRSBF — Worker automático Colab ↔ GitHub

**Você só precisa executar a célula abaixo uma vez.**

Ela conecta ao GitHub usando o secret `GITHUB_TOKEN`, clona/atualiza o projeto, verifica a GPU e inicia o worker contínuo. Enquanto a sessão do Colab estiver viva, novos jobs em `jobs/queue.json` serão executados automaticamente e os resultados voltarão ao GitHub.

In [ ]:
from google.colab import userdata
import os, pathlib, subprocess, sys

REPO = 'https://github.com/contatofilipevono-rgb/hrsbf-bent-verification.git'
ROOT = pathlib.Path('/content/hrsbf-bent-verification')

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Secret GITHUB_TOKEN não encontrado ou sem acesso ao notebook.')

print('1/4 Conectando ao GitHub...')
subprocess.run(['gh','auth','login','--hostname','github.com','--with-token'], input=token, text=True, check=True)
subprocess.run(['gh','auth','setup-git'], check=True)

print('2/4 Sincronizando repositório...')
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone',REPO,str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--rebase','origin','main'], check=True)

os.chdir(ROOT)
print('3/4 Verificando GPU...')
gpu = subprocess.run(['nvidia-smi','-L'], text=True, capture_output=True)
print(gpu.stdout.strip() if gpu.returncode == 0 else 'GPU não detectada. Ative uma GPU no runtime do Colab.')

print('4/4 Iniciando worker contínuo...')
print('Deixe esta célula rodando enquanto quiser usar o Colab como executor remoto.')
subprocess.run([sys.executable,'colab_worker.py','--poll','30'], check=True)
